In [26]:
import pandas as pd
import numpy as np
from functions import scrape_nba_data
import matplotlib.pyplot as plt

In [27]:
raw_df = scrape_nba_data(2021, 2026)
raw_df.head()

Scraping 2021-22...
  605 players
Scraping 2022-23...
  539 players
Scraping 2023-24...
  572 players
Scraping 2024-25...
  569 players
Scraping 2025-26...
  582 players
Scraping 2026-27...
  0 players

Saved 2867 rows to nba_data.csv


/Users/treychase/nets_project/nets-project/functions.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(all_seasons, ignore_index=True)


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,DEF_RATING,NET_RATING,TS_PCT,EFG_PCT,USG_PCT,AST_PCT,REB_PCT,PIE,PACE,SEASON
0,203932,Aaron Gordon,Aaron,1610612743,DEN,26.0,75,46,29,0.613,...,111.1,6.2,0.602,0.573,0.194,0.109,0.095,0.096,99.00,2021-22
1,1630565,Aaron Henry,Aaron,1610612755,PHI,22.0,6,6,0,1.000,...,127.0,-56.0,0.200,0.200,0.149,0.000,0.024,-0.079,105.99,2021-22
2,1628988,Aaron Holiday,Aaron,1610612756,PHX,25.0,63,34,29,0.540,...,108.2,-1.4,0.544,0.504,0.182,0.223,0.058,0.088,99.57,2021-22
3,1630174,Aaron Nesmith,Aaron,1610612738,BOS,22.0,52,32,20,0.615,...,104.3,3.0,0.507,0.481,0.164,0.057,0.073,0.048,100.58,2021-22
4,1630598,Aaron Wiggins,Aaron,1610612760,OKC,23.0,50,13,37,0.260,...,112.6,-8.1,0.556,0.525,0.149,0.083,0.069,0.062,101.60,2021-22


In [28]:
raw_df.describe()

,AGE,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OFF_RATING,DEF_RATING,NET_RATING,TS_PCT,EFG_PCT,USG_PCT,AST_PCT,REB_PCT,PIE,PACE
count,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.00000,2867.000000,2867.000000,...,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000,2867.000000
mean,26.206488,0.486667,19.281374,3.214719,6.937531,0.450222,0.987618,2.81158,0.300759,1.343739,...,108.890548,111.341367,-2.450192,0.547487,0.518172,0.177115,0.141106,0.088196,0.086758,101.561448
std,4.230029,0.198931,9.539865,2.360834,4.856586,0.118182,0.878631,2.27472,0.142688,1.406487,...,11.436216,9.670114,15.040207,0.117193,0.120931,0.056880,0.090286,0.042502,0.044560,3.949219
min,19.000000,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,-250.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.286000,70.430000
25%,23.000000,0.351500,11.600000,1.400000,3.300000,0.406000,0.300000,1.00000,0.263000,0.400000,...,105.850000,108.400000,-7.300000,0.517000,0.485000,0.139000,0.078000,0.058500,0.066500,99.355000
50%,25.000000,0.506000,18.900000,2.600000,5.700000,0.451000,0.800000,2.40000,0.336000,0.900000,...,110.400000,111.700000,-1.100000,0.565000,0.533000,0.169000,0.117000,0.078000,0.086000,101.170000
75%,29.000000,0.617000,27.300000,4.400000,9.400000,0.500000,1.500000,4.20000,0.380000,1.700000,...,114.100000,115.100000,3.700000,0.604000,0.574000,0.208000,0.190000,0.111000,0.109000,103.080000
max,43.000000,1.000000,43.500000,11.800000,23.600000,1.000000,4.900000,11.80000,1.000000,10.200000,...,200.000000,250.000000,133.300000,1.250000,1.250000,0.500000,1.000000,0.500000,0.519000,142.290000


In [29]:
raw_df.dtypes

PLAYER_ID             object
PLAYER_NAME           object
NICKNAME              object
TEAM_ID               object
TEAM_ABBREVIATION     object
                      ...   
AST_PCT              float64
REB_PCT              float64
PIE                  float64
PACE                 float64
SEASON                object
Length: 78, dtype: object

In [30]:
# Some players get traded mid-season: the API gives one row per team plus a
# "TOT" (total) row combining them. Keep only the TOT row when present so
# each player has exactly one row per season. (A groupby(...).apply() here
# would silently drop PLAYER_ID/SEASON under current pandas, since group-by
# columns are excluded from what's passed to the function — so this uses a
# vectorized transform instead.)
has_tot = raw_df.groupby(["PLAYER_ID", "SEASON"])["TEAM_ABBREVIATION"].transform(
    lambda s: (s == "TOT").any()
)
df = raw_df[~has_tot | (raw_df["TEAM_ABBREVIATION"] == "TOT")].reset_index(drop=True)

# eda_box_scores.py expects the scraper's original column names (SEASON,
# PLAYER_ID, FG_PCT, GP, MIN, FGA, ...), so the deduplicated data is saved
# under those names — this is the file the EDA cell below reads.
df.to_csv("nba_data.csv", index=False)

# A friendlier, renamed per-game table for display/export. Not used by the
# EDA script, which relies on the original column names saved above.
id_cols = ["SEASON", "PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "PLAYER_POSITION"]

shooting_cols = ["FG_PCT", "FG3_PCT", "FT_PCT"]

per_game_cols = ["PTS", "REB", "AST", "STL", "TOV"]

advanced_cols = [
    "OFF_RATING", "DEF_RATING", "NET_RATING",
    "TS_PCT", "EFG_PCT", "USG_PCT", "AST_PCT", "REB_PCT", "PIE", "PACE",
]

keep_cols = id_cols + shooting_cols + per_game_cols + advanced_cols
keep_cols = [c for c in keep_cols if c in df.columns]

per_game = df[keep_cols].rename(columns={
    "SEASON": "YEAR",
    "PLAYER_NAME": "PLAYER",
    "TEAM_ABBREVIATION": "TEAM",
    "PTS": "PPG",
    "REB": "RPG",
    "AST": "APG",
    "STL": "SPG",
    "TOV": "TOPG",
})

per_game = per_game.sort_values(["YEAR", "PLAYER"]).reset_index(drop=True)

per_game.to_csv("nba_per_game.csv", index=False)
per_game.head()


,YEAR,PLAYER,TEAM,AGE,FG_PCT,FG3_PCT,FT_PCT,PPG,RPG,APG,...,OFF_RATING,DEF_RATING,NET_RATING,TS_PCT,EFG_PCT,USG_PCT,AST_PCT,REB_PCT,PIE,PACE
0,2021-22,Aaron Gordon,DEN,26.0,0.520,0.335,0.743,15.0,5.9,2.5,...,117.3,111.1,6.2,0.602,0.573,0.194,0.109,0.095,0.096,99.00
1,2021-22,Aaron Henry,PHI,22.0,0.200,0.000,0.000,0.3,0.2,0.0,...,71.1,127.0,-56.0,0.200,0.200,0.149,0.000,0.024,-0.079,105.99
2,2021-22,Aaron Holiday,PHX,25.0,0.447,0.379,0.868,6.3,1.9,2.4,...,106.8,108.2,-1.4,0.544,0.504,0.182,0.223,0.058,0.088,99.57
3,2021-22,Aaron Nesmith,BOS,22.0,0.396,0.270,0.808,3.8,1.7,0.4,...,107.3,104.3,3.0,0.507,0.481,0.164,0.057,0.073,0.048,100.58
4,2021-22,Aaron Wiggins,OKC,23.0,0.463,0.304,0.729,8.3,3.6,1.4,...,104.5,112.6,-8.1,0.556,0.525,0.149,0.083,0.069,0.062,101.60


In [31]:
from eda_box_scores import run_eda

run_eda(input_csv="nba_data.csv", output_dir="eda_output")

Loaded 2867 player-season rows, 5 seasons (2021-22 to 2025-26)

=== MISSINGNESS ===
Series([], )

=== DISTRIBUTIONS ===
         FG_PCT   FG3_PCT    FT_PCT    TS_PCT   EFG_PCT       AGE   USG_PCT       MIN        GP
count  2867.000  2867.000  2867.000  2867.000  2867.000  2867.000  2867.000  2867.000  2867.000
mean      0.450     0.301     0.710     0.547     0.518    26.206     0.177    19.281    45.794
std       0.118     0.143     0.223     0.117     0.121     4.230     0.057     9.540    25.114
min       0.000     0.000     0.000     0.000     0.000    19.000     0.000     0.400     1.000
25%       0.406     0.263     0.667     0.517     0.485    23.000     0.139    11.600    23.000
50%       0.451     0.336     0.763     0.565     0.533    25.000     0.169    18.900    51.000
75%       0.500     0.380     0.834     0.604     0.574    29.000     0.208    27.300    68.000
max       1.000     1.000     1.000     1.250     1.250    43.000     0.500    43.500    84.000

20.3% of player

/Users/treychase/nets_project/nets-project/eda_box_scores.py:138: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  binned = valid.groupby(pd.cut(valid["AGE"], bins=range(18, 42, 2)))["TS_PCT"].mean()



Variance Inflation Factors (VIF > ~5 signals a collinearity problem):
 column    VIF
 FG_PCT 204.34
FG3_PCT  11.43
 FT_PCT  17.56
 TS_PCT 435.86
EFG_PCT 522.63

=== AUTOCORRELATION (lag-1 TS%) ===
Lag-1 (consecutive-season) TS% correlation: 0.551
Player-seasons with a valid consecutive prior season: 1469 of 2286 total (64.3%)

=== VARIANCE DECOMPOSITION ===
Between-player variance share: 71.7%
Within-player (season-to-season) variance share: 28.3%
A large between-player share supports a hierarchical model with player-level intercepts rather than pooling everyone together.

=== SAMPLE SIZE / ATTRITION ===
Mean TS% for player-seasons WITH vs WITHOUT a following season (a gap here suggests non-random attrition):
HAS_NEXT_SEASON
False    0.520
True     0.564

62.6% of player-seasons have a usable next season for the AR target.

Saved plots and lagged pairs dataset to eda_output/


## Bayesian Gaussian AR(3) shooting model

Projects a player's true shooting percentage (TS%) from their age, up to
three most recent prior-season TS% values, and usage rate (USG%). The spec
below is informed directly by the EDA above rather than guessed:

- **Flexible AR(3) term** on the last three seasons of TS% -- rather than
  assume persistence decays after one lag, or require a full three-season
  history before a player can be modeled at all, each lag is only used when
  that exact season actually exists in the data (checked against the season
  gap, not just "the previous row," so an injury/G-League year isn't
  mistaken for a valid lag). Missing lags are imputed to the population
  mean TS% (so they contribute nothing to the prediction, not garbage), and
  an `n_lags_available` covariate lets the model learn a systematic
  adjustment for players with less history behind them -- so a rookie's
  first qualifying season is modeled from age and usage alone, a 2nd-year
  player gets one lag, and so on, rather than being excluded outright.
- **Age + age²** -- the age-vs-TS% relationship above is curved, not linear.
- **Usage rate** -- TS% and USG% trade off within a season (higher-usage
  players tend to take harder, lower-efficiency shots), so USG% is included
  as a covariate alongside the AR terms.
- **Gaussian likelihood** -- TS% is roughly bell-shaped and continuous once
  the low-attempts players are filtered out (same floor as the EDA above).

Requirements: `pip install pymc arviz`

In [33]:
pip install pymc arviz

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.9/562.9 kB 5.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.3 MB/s eta 0:00:00a 0:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 20.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 13.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.7/97.7 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 39.2 MB/

In [ ]:
import os

import pymc as pm
import arviz as az

from eda_box_scores import load_data

MIN_SEASON_FGA = 50  # same attempts floor used in the EDA distribution/autocorrelation checks

model_source = load_data("nba_data.csv")

ar_cols = ["PLAYER_ID", "PLAYER_NAME", "SEASON_START_YEAR", "AGE", "TS_PCT", "USG_PCT", "SEASON_FGA_EST"]
ar_df = (
    model_source[ar_cols]
    .dropna()
    .sort_values(["PLAYER_ID", "SEASON_START_YEAR"])
)
ar_df = ar_df[ar_df["SEASON_FGA_EST"] >= MIN_SEASON_FGA].reset_index(drop=True)

# Flexible lags: every player-season is usable, not just ones with a full
# 3-season history. PREV_TS_PCT_k is only trusted when the season exactly k
# years earlier is actually present -- shift(k) alone can't tell a true lag
# apart from an injury/G-League gap, so it's checked against SEASON_START_YEAR.
for k in (1, 2, 3):
    ar_df[f"PREV_TS_PCT_{k}"] = ar_df.groupby("PLAYER_ID")["TS_PCT"].shift(k)
    season_lag = ar_df.groupby("PLAYER_ID")["SEASON_START_YEAR"].shift(k)
    ar_df[f"HAS_LAG_{k}"] = (ar_df["SEASON_START_YEAR"] - season_lag) == k

ar_df["N_LAGS_AVAILABLE"] = ar_df[["HAS_LAG_1", "HAS_LAG_2", "HAS_LAG_3"]].sum(axis=1)
ar_model_df = ar_df

print(f"{len(ar_model_df)} player-seasons total (every one usable regardless of history length)")
print(ar_model_df["N_LAGS_AVAILABLE"].value_counts().sort_index().rename("player-seasons by lags available"))
ar_model_df.head()

In [ ]:
# Standardize predictors so the priors below are on a comparable, weakly-informative
# scale. All three lags are the same underlying quantity (TS_PCT) at different
# offsets, so they share one scale (fit on the target y) instead of each getting
# their own mean/sd.
y = ar_model_df["TS_PCT"].to_numpy()
age = ar_model_df["AGE"].to_numpy()
usg = ar_model_df["USG_PCT"].to_numpy()

ts_mean, ts_sd = y.mean(), y.std()
age_mean, age_sd = age.mean(), age.std()
usg_mean, usg_sd = usg.mean(), usg.std()


def lagged_z(k):
    """Standardized lag-k TS%, imputed to the population mean (z-score 0) for
    player-seasons where that lag isn't available -- contributes nothing to
    mu rather than being dropped from the model."""
    raw = ar_model_df[f"PREV_TS_PCT_{k}"].to_numpy()
    has = ar_model_df[f"HAS_LAG_{k}"].to_numpy()
    filled = np.where(has, raw, ts_mean)
    return (filled - ts_mean) / ts_sd


age_z = (age - age_mean) / age_sd
usg_z = (usg - usg_mean) / usg_sd
prev1_z = lagged_z(1)
prev2_z = lagged_z(2)
prev3_z = lagged_z(3)

n_lags = ar_model_df["N_LAGS_AVAILABLE"].to_numpy()
n_lags_mean, n_lags_sd = n_lags.mean(), n_lags.std()
n_lags_z = (n_lags - n_lags_mean) / n_lags_sd

In [ ]:
with pm.Model() as ts_ar_model:
    alpha = pm.Normal("alpha", mu=y.mean(), sigma=0.2)
    phi1 = pm.Normal("phi1", mu=0, sigma=1)
    phi2 = pm.Normal("phi2", mu=0, sigma=1)
    phi3 = pm.Normal("phi3", mu=0, sigma=1)
    beta_age = pm.Normal("beta_age", mu=0, sigma=1)
    beta_age_sq = pm.Normal("beta_age_sq", mu=0, sigma=1)
    beta_usg = pm.Normal("beta_usg", mu=0, sigma=1)
    beta_n_lags = pm.Normal("beta_n_lags", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=0.2)

    mu = (
        alpha
        + phi1 * prev1_z
        + phi2 * prev2_z
        + phi3 * prev3_z
        + beta_age * age_z
        + beta_age_sq * age_z**2
        + beta_usg * usg_z
        + beta_n_lags * n_lags_z
    )
    pm.Normal("ts_pct_obs", mu=mu, sigma=sigma, observed=y)

    ar_idata = pm.sample(2000, tune=2000, chains=4, target_accept=0.9, random_seed=42)
    ar_idata.extend(pm.sample_posterior_predictive(ar_idata, random_seed=42))
    pm.compute_log_likelihood(ar_idata)

In [ ]:
AR_VAR_NAMES = ["alpha", "phi1", "phi2", "phi3", "beta_age", "beta_age_sq", "beta_usg", "beta_n_lags", "sigma"]

ar_summary = az.summary(ar_idata, var_names=AR_VAR_NAMES, round_to=4)
print(ar_summary)

n_divergent = int(ar_idata.sample_stats["diverging"].sum())
n_draws = ar_idata.sample_stats.sizes["chain"] * ar_idata.sample_stats.sizes["draw"]
print(f"\nDivergent transitions: {n_divergent} / {n_draws}")
print(f"Max R-hat: {ar_summary['r_hat'].max():.4f} (should be close to 1.00)")
print(f"Min ESS (bulk): {ar_summary['ess_bulk'].min():.0f}")
print(f"Min ESS (tail): {ar_summary['ess_tail'].min():.0f}")

In [ ]:
os.makedirs("bayesian_ar_output", exist_ok=True)

az.plot_trace(ar_idata, var_names=AR_VAR_NAMES, compact=True)
plt.tight_layout()
plt.savefig("bayesian_ar_output/trace_plots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Rank plots: a healthy chain mix looks roughly uniform across ranks, with no chain
# systematically higher or lower than the others
az.plot_rank(ar_idata, var_names=AR_VAR_NAMES)
plt.tight_layout()
plt.savefig("bayesian_ar_output/rank_plots.png", dpi=150)
plt.show()

# Energy plot: a healthy NUTS run has the marginal and transition energy
# distributions overlapping closely
az.plot_energy(ar_idata)
plt.tight_layout()
plt.savefig("bayesian_ar_output/energy_plot.png", dpi=150)
plt.show()

az.plot_forest(ar_idata, var_names=AR_VAR_NAMES, combined=True, hdi_prob=0.94)
plt.tight_layout()
plt.savefig("bayesian_ar_output/forest_plot.png", dpi=150)
plt.show()

In [ ]:
# Posterior predictive check: the observed TS% distribution should sit
# comfortably inside the spread of the posterior predictive draws
az.plot_ppc(ar_idata, num_pp_samples=200)
plt.tight_layout()
plt.savefig("bayesian_ar_output/posterior_predictive_check.png", dpi=150)
plt.show()

pp_samples = ar_idata.posterior_predictive["ts_pct_obs"].values.reshape(-1, len(y))
bayes_r2 = az.r2_score(y, pp_samples)
print(f"Bayesian R^2: {bayes_r2['r2']:.3f} (sd {bayes_r2['r2_std']:.3f})")

# LOO-CV: Pareto k values above ~0.7 flag observations the model struggles to predict
loo = az.loo(ar_idata, pointwise=True)
print(f"\n{loo}")

az.plot_khat(loo)
plt.tight_layout()
plt.savefig("bayesian_ar_output/pareto_k_diagnostic.png", dpi=150)
plt.show()

In [ ]:
pred_mean = ar_idata.posterior_predictive["ts_pct_obs"].mean(dim=("chain", "draw")).values

plt.figure(figsize=(6, 6))
plt.scatter(y, pred_mean, alpha=0.4, s=15)
lims = [min(y.min(), pred_mean.min()), max(y.max(), pred_mean.max())]
plt.plot(lims, lims, "r--", linewidth=1)
plt.xlabel("Observed TS%")
plt.ylabel("Posterior predictive mean TS%")
plt.title("Observed vs. predicted TS%")
plt.tight_layout()
plt.savefig("bayesian_ar_output/observed_vs_predicted.png", dpi=150)
plt.show()

In [ ]:
def project_ts_pct(age_value, usg_pct_value, prev_ts_pct_1=None, prev_ts_pct_2=None, prev_ts_pct_3=None, idata=ar_idata):
    """Posterior samples of projected TS% for a given age and usage rate, using
    whichever of the three most recent prior-season TS% values are available
    (pass None, or omit, for any that don't exist -- e.g. a 2nd-year player has
    no prev_ts_pct_2/3). Missing lags are imputed the same way as in training:
    treated as the population mean, with n_lags_available capturing how much
    history is actually behind the projection."""
    post = idata.posterior
    age_z_ = (age_value - age_mean) / age_sd
    usg_z_ = (usg_pct_value - usg_mean) / usg_sd

    raw_lags = [prev_ts_pct_1, prev_ts_pct_2, prev_ts_pct_3]
    n_avail = sum(v is not None for v in raw_lags)
    prev1_z_, prev2_z_, prev3_z_ = (
        ((v if v is not None else ts_mean) - ts_mean) / ts_sd for v in raw_lags
    )
    n_lags_z_ = (n_avail - n_lags_mean) / n_lags_sd

    mu_samples = (
        post["alpha"]
        + post["phi1"] * prev1_z_
        + post["phi2"] * prev2_z_
        + post["phi3"] * prev3_z_
        + post["beta_age"] * age_z_
        + post["beta_age_sq"] * age_z_**2
        + post["beta_usg"] * usg_z_
        + post["beta_n_lags"] * n_lags_z_
    ).values.flatten()
    return mu_samples


example_samples = project_ts_pct(
    age_value=27, usg_pct_value=0.24, prev_ts_pct_1=0.58, prev_ts_pct_2=0.56, prev_ts_pct_3=0.55
)
example_hdi = az.hdi(example_samples, hdi_prob=0.94)
print("Projected TS% for a 27-year-old with a 58/56/55% prior three-season TS% and 24% usage:")
print(f"  mean = {example_samples.mean():.3f}, 94% HDI = [{example_hdi[0]:.3f}, {example_hdi[1]:.3f}]")

rookie_samples = project_ts_pct(age_value=20, usg_pct_value=0.20)
rookie_hdi = az.hdi(rookie_samples, hdi_prob=0.94)
print("\nProjected TS% for a 20-year-old with no prior NBA seasons and 20% usage:")
print(f"  mean = {rookie_samples.mean():.3f}, 94% HDI = [{rookie_hdi[0]:.3f}, {rookie_hdi[1]:.3f}]")

### Reading the diagnostics

- **Convergence** -- `r_hat` should be at (or very close to) 1.00 and ESS
  (bulk/tail) should be in the low thousands or better; either one off
  signals the chains haven't mixed and `ar_idata` shouldn't be trusted.
- **Divergences** -- any divergent transitions point to regions of the
  posterior NUTS couldn't sample reliably (often a sign priors/parameterization
  need reworking), independent of what `r_hat`/ESS say.
- **Trace/rank/energy plots** -- traces should look like stationary "fuzzy
  caterpillars" with chains overlapping; rank plots should be roughly
  uniform; the energy plot's marginal and transition distributions should
  overlap closely.
- **Posterior predictive check / Bayesian R²** -- how well the fitted model
  reproduces the observed TS% distribution and values.
- **Pareto k (LOO-CV)** -- flags individual player-seasons the model can't
  predict well out-of-sample; a cluster of high-k points usually means a
  missing covariate or a form the Gaussian likelihood doesn't capture.
- **`phi1`/`phi2`/`phi3`** -- the weight on each of the last three seasons,
  when available. A smooth decay (phi1 largest, phi3 smallest) is the usual
  "recent seasons matter more" pattern, but it isn't guaranteed -- a
  farther-back lag can legitimately come out larger than a more recent one.
  Check each one's HDI against zero before reading much into the ordering:
  a real but small non-monotonic pattern is still a real pattern, not
  automatically noise, but it's also not strong evidence of anything if the
  HDI is wide relative to the difference between lags.
- **`beta_n_lags`** -- whether having more (or less) career history behind a
  player-season systematically shifts predicted TS%, independent of age. A
  clearly nonzero HDI here means the model has learned something real about
  early-career vs. established players beyond what age alone captures; an
  HDI straddling zero means it hasn't found that distinction useful.

**Note on flexible lags:** every player-season is included here regardless
of how much history exists behind it -- a rookie's first qualifying season
contributes with all three lags imputed to the population mean (so age and
usage do the work), and `N_LAGS_AVAILABLE` lets the model separately learn
whether that matters. This trades the all-or-nothing AR(3) filter (which
would exclude anyone without three consecutive prior seasons) for a model
that uses everyone but is explicit about how much it actually knows about
each row.

## Hierarchical extension: player-level intercepts

The EDA's variance-decomposition check flagged a large between-player
variance share as evidence for player-level random effects, but with only
one lag-1 transition per player that structure wasn't identifiable. Now
that 2021-22 through 2025-26 are scraped -- and every player-season is
usable regardless of history length -- this section adds a **non-centered
partial-pooling intercept per player** on top of the pooled flexible-lag
AR(3) model above:

```
TS%ᵢ,ₜ ~ Normal(αᵢ + φ1·TS%ₜ₋₁ + φ2·TS%ₜ₋₂ + φ3·TS%ₜ₋₃ + β_age·age + β_age²·age² + β_usg·usg + β_lags·n_lags, σ)
αᵢ = μ_α + τ_α · offsetᵢ,   offsetᵢ ~ Normal(0, 1)
```

`μ_α` and `τ_α` are the population-level mean intercept and the
between-player spread around it -- the same non-centered trick used to
avoid Neal's-funnel divergences in any hierarchical model. This section
doesn't assume the extra complexity is worth it: it fits the hierarchical
model, then compares it against the pooled model above with LOO-CV rather
than taking partial pooling on faith.

In [ ]:
player_codes, player_uniques = pd.factorize(ar_model_df["PLAYER_ID"])
player_uniques = np.asarray(player_uniques)
n_players = len(player_uniques)

print(f"{n_players} unique players across {len(ar_model_df)} player-seasons")

hier_coords = {"player": player_uniques, "obs": ar_model_df.index}

In [ ]:
with pm.Model(coords=hier_coords) as ts_ar_hier_model:
    player_idx = pm.Data("player_idx", player_codes, dims="obs")

    mu_alpha = pm.Normal("mu_alpha", mu=y.mean(), sigma=0.2)
    tau_alpha = pm.HalfNormal("tau_alpha", sigma=0.1)
    alpha_offset = pm.Normal("alpha_offset", mu=0, sigma=1, dims="player")
    alpha_player = pm.Deterministic("alpha_player", mu_alpha + tau_alpha * alpha_offset, dims="player")

    phi1 = pm.Normal("phi1", mu=0, sigma=1)
    phi2 = pm.Normal("phi2", mu=0, sigma=1)
    phi3 = pm.Normal("phi3", mu=0, sigma=1)
    beta_age = pm.Normal("beta_age", mu=0, sigma=1)
    beta_age_sq = pm.Normal("beta_age_sq", mu=0, sigma=1)
    beta_usg = pm.Normal("beta_usg", mu=0, sigma=1)
    beta_n_lags = pm.Normal("beta_n_lags", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=0.2)

    mu = (
        alpha_player[player_idx]
        + phi1 * prev1_z
        + phi2 * prev2_z
        + phi3 * prev3_z
        + beta_age * age_z
        + beta_age_sq * age_z**2
        + beta_usg * usg_z
        + beta_n_lags * n_lags_z
    )
    pm.Normal("ts_pct_obs", mu=mu, sigma=sigma, dims="obs", observed=y)

    # Hierarchical models need a higher target_accept than the pooled model to avoid
    # divergences from the non-centered funnel geometry
    hier_idata = pm.sample(2000, tune=2000, chains=4, target_accept=0.98, random_seed=42)
    hier_idata.extend(pm.sample_posterior_predictive(hier_idata, random_seed=42))
    pm.compute_log_likelihood(hier_idata)

In [ ]:
HIER_POP_VAR_NAMES = ["mu_alpha", "tau_alpha", "phi1", "phi2", "phi3", "beta_age", "beta_age_sq", "beta_usg", "beta_n_lags", "sigma"]

hier_summary = az.summary(hier_idata, var_names=HIER_POP_VAR_NAMES, round_to=4)
print(hier_summary)

n_divergent_hier = int(hier_idata.sample_stats["diverging"].sum())
n_draws_hier = hier_idata.sample_stats.sizes["chain"] * hier_idata.sample_stats.sizes["draw"]
print(f"\nDivergent transitions: {n_divergent_hier} / {n_draws_hier}")
print(f"Max R-hat: {hier_summary['r_hat'].max():.4f} (should be close to 1.00)")
print(f"Min ESS (bulk): {hier_summary['ess_bulk'].min():.0f}")
print(f"Min ESS (tail): {hier_summary['ess_tail'].min():.0f}")

# Per-player intercepts are checked in aggregate rather than printed individually --
# there's one for every player, so a table of 300+ rows isn't useful here
player_summary = az.summary(hier_idata, var_names=["alpha_player"], round_to=4)
print(f"\nPer-player intercepts: {len(player_summary)} players")
print(f"Max R-hat across player intercepts: {player_summary['r_hat'].max():.4f}")
print(f"Min ESS (bulk) across player intercepts: {player_summary['ess_bulk'].min():.0f}")

In [ ]:
az.plot_trace(hier_idata, var_names=HIER_POP_VAR_NAMES, compact=True)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_trace_plots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
az.plot_rank(hier_idata, var_names=HIER_POP_VAR_NAMES)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_rank_plots.png", dpi=150)
plt.show()

az.plot_energy(hier_idata)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_energy_plot.png", dpi=150)
plt.show()

In [ ]:
az.plot_ppc(hier_idata, num_pp_samples=200)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_posterior_predictive_check.png", dpi=150)
plt.show()

hier_pp_samples = hier_idata.posterior_predictive["ts_pct_obs"].values.reshape(-1, len(y))
hier_bayes_r2 = az.r2_score(y, hier_pp_samples)
print(f"Bayesian R^2 (hierarchical): {hier_bayes_r2['r2']:.3f} (sd {hier_bayes_r2['r2_std']:.3f})")

hier_loo = az.loo(hier_idata, pointwise=True)
print(f"\n{hier_loo}")

az.plot_khat(hier_loo)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_pareto_k_diagnostic.png", dpi=150)
plt.show()

In [ ]:
# The deciding comparison: does partial pooling actually improve out-of-sample
# fit over the simpler pooled model, or just add unneeded complexity?
model_comparison = az.compare({"pooled": ar_idata, "hierarchical": hier_idata})
print(model_comparison)

az.plot_compare(model_comparison)
plt.tight_layout()
plt.savefig("bayesian_ar_output/model_comparison.png", dpi=150)
plt.show()

In [ ]:
player_obs_counts = ar_model_df.groupby("PLAYER_ID").size()
player_emp_mean = ar_model_df.groupby("PLAYER_ID")["TS_PCT"].mean()
player_post_mean = pd.Series(
    hier_idata.posterior["alpha_player"].mean(dim=("chain", "draw")).values,
    index=player_uniques,
)

shrinkage_df = pd.DataFrame({
    "n_obs": player_obs_counts,
    "empirical_mean_ts": player_emp_mean,
    "posterior_mean_intercept": player_post_mean,
}).dropna()

plt.figure(figsize=(7, 6))
sc = plt.scatter(
    shrinkage_df["empirical_mean_ts"],
    shrinkage_df["posterior_mean_intercept"],
    c=shrinkage_df["n_obs"],
    cmap="viridis",
    alpha=0.7,
    s=25,
)
lims = [
    shrinkage_df[["empirical_mean_ts", "posterior_mean_intercept"]].min().min(),
    shrinkage_df[["empirical_mean_ts", "posterior_mean_intercept"]].max().max(),
]
plt.plot(lims, lims, "r--", linewidth=1, label="no shrinkage")
plt.colorbar(sc, label="player-seasons observed")
plt.xlabel("Empirical mean TS% (raw)")
plt.ylabel("Posterior mean player intercept")
plt.title("Partial-pooling shrinkage toward the population mean")
plt.legend()
plt.tight_layout()
plt.savefig("bayesian_ar_output/shrinkage_plot.png", dpi=150)
plt.show()

In [ ]:
def project_ts_pct_hier(age_value, usg_pct_value, prev_ts_pct_1=None, prev_ts_pct_2=None, prev_ts_pct_3=None, player_id=None, idata=hier_idata):
    """Posterior samples of projected TS%, using a player's own partial-pooled
    intercept when player_id is a player seen in ar_model_df, otherwise drawing
    from the population-level intercept distribution (e.g. an incoming rookie).
    prev_ts_pct_1/2/3 are that player's most recent seasons, most recent first --
    pass None (or omit) for any that don't exist."""
    post = idata.posterior
    age_z_ = (age_value - age_mean) / age_sd
    usg_z_ = (usg_pct_value - usg_mean) / usg_sd

    raw_lags = [prev_ts_pct_1, prev_ts_pct_2, prev_ts_pct_3]
    n_avail = sum(v is not None for v in raw_lags)
    prev1_z_, prev2_z_, prev3_z_ = (
        ((v if v is not None else ts_mean) - ts_mean) / ts_sd for v in raw_lags
    )
    n_lags_z_ = (n_avail - n_lags_mean) / n_lags_sd

    if player_id is not None and player_id in player_uniques:
        p_idx = int(np.where(player_uniques == player_id)[0][0])
        alpha_samples = post["alpha_player"].isel(player=p_idx)
    else:
        rng = np.random.default_rng(0)
        alpha_samples = post["mu_alpha"] + post["tau_alpha"] * rng.standard_normal(post["mu_alpha"].shape)

    mu_samples = (
        alpha_samples
        + post["phi1"] * prev1_z_
        + post["phi2"] * prev2_z_
        + post["phi3"] * prev3_z_
        + post["beta_age"] * age_z_
        + post["beta_age_sq"] * age_z_**2
        + post["beta_usg"] * usg_z_
        + post["beta_n_lags"] * n_lags_z_
    ).values.flatten()
    return mu_samples


example_player = ar_model_df["PLAYER_ID"].iloc[0]
hier_samples = project_ts_pct_hier(
    age_value=27, usg_pct_value=0.24, prev_ts_pct_1=0.58, prev_ts_pct_2=0.56, prev_ts_pct_3=0.55, player_id=example_player
)
hier_hdi = az.hdi(hier_samples, hdi_prob=0.94)
print(f"Projected TS% for player {example_player} (partial-pooled intercept):")
print(f"  mean = {hier_samples.mean():.3f}, 94% HDI = [{hier_hdi[0]:.3f}, {hier_hdi[1]:.3f}]")

new_player_samples = project_ts_pct_hier(age_value=20, usg_pct_value=0.20, player_id=None)
new_player_hdi = az.hdi(new_player_samples, hdi_prob=0.94)
print("\nProjected TS% for a rookie not in the training data, with no prior NBA seasons:")
print(f"  mean = {new_player_samples.mean():.3f}, 94% HDI = [{new_player_hdi[0]:.3f}, {new_player_hdi[1]:.3f}]")

### Reading the hierarchical extension

- **`mu_alpha` / `tau_alpha`** -- the population-level mean intercept and the
  between-player standard deviation around it. A `tau_alpha` posterior
  concentrated near zero means the data don't support much player-to-player
  variation beyond what the lagged TS% terms, age, usage, and lag
  availability already explain -- the AR terms are themselves a strong
  proxy for "who this player is," so don't expect this to match the EDA's
  raw between-player variance share, which was computed without
  conditioning on prior seasons.
- **Per-player intercept diagnostics** -- checked in aggregate (max r-hat,
  min ESS) rather than printed individually. A handful of players near a
  divergence or with low ESS is normal (usually a player with very few
  observations); a widespread pattern signals a convergence issue with the
  whole hierarchy, not just those players.
- **Model comparison (`az.compare`)** -- the deciding evidence for whether
  the added complexity is worth it, *but check the `warning` column first*.
  Flexible lags mean many player-seasons are a player's only observation in
  the window (zero lags available -- often a veteran whose career mostly
  predates 2021-22). Leaving out someone's *only* data point removes nearly
  all information about their intercept, which is exactly the situation
  where PSIS-LOO's importance-sampling approximation breaks down: expect a
  Pareto-k warning here, concentrated on those single-observation players,
  and don't take a "hierarchical wins" verdict at face value if it's
  flagged unreliable -- the pooled model's LOO estimate doesn't have this
  problem and is the safer comparison to trust.
- **Shrinkage plot** -- points near the red no-shrinkage line have enough of
  their own data that the model trusts their empirical mean; points pulled
  toward the population mean (especially low-observation-count ones) are
  exactly the partial-pooling behavior a hierarchical model is for -- a
  rookie or injury-shortened player shouldn't be judged purely on a handful
  of noisy games.
- **New-player projections** -- `project_ts_pct_hier` falls back to sampling
  from the population-level intercept distribution for any `player_id` not
  in `ar_model_df`, and now also accepts a player with zero prior TS% at
  all (a true rookie), since lags are flexible rather than required.

## Next-season dashboard

Projects every currently-active player's TS% for the season immediately
after the last one scraped, using the hierarchical model above, then
renders a sorted, searchable HTML dashboard via `dashboard.py`.

Because lags are flexible, this covers every player who logged a
qualifying season most recently -- including rookies and short-tenured
players who'd have been excluded by a strict "three consecutive seasons"
requirement. One thing worth being explicit about before trusting the
ranking: **usage is held at its most recent observed value.** Next
season's actual usage rate is unknown at projection time, so this is a
steady-state assumption, not a usage forecast -- a player about to see a
much bigger (or smaller) role isn't reflected here.

In [ ]:
proj_source = (
    model_source[model_source["SEASON_FGA_EST"] >= MIN_SEASON_FGA]
    .dropna(subset=["AGE", "TS_PCT", "USG_PCT"])
    .sort_values(["PLAYER_ID", "SEASON_START_YEAR"])
    .reset_index(drop=True)
)
proj_source["TS_PCT_LAG_1"] = proj_source.groupby("PLAYER_ID")["TS_PCT"].shift(1)
proj_source["TS_PCT_LAG_2"] = proj_source.groupby("PLAYER_ID")["TS_PCT"].shift(2)
proj_source["SEASON_LAG_1"] = proj_source.groupby("PLAYER_ID")["SEASON_START_YEAR"].shift(1)
proj_source["SEASON_LAG_2"] = proj_source.groupby("PLAYER_ID")["SEASON_START_YEAR"].shift(2)

latest_season_year = proj_source["SEASON_START_YEAR"].max()
next_season_rows = proj_source[proj_source["SEASON_START_YEAR"] == latest_season_year].copy()

# Flexible lags for the projection step: every active player gets a projection --
# their own most recent season always counts as lag 1, and lags 2/3 are added
# only when that exact season is present (not just "the previous row"), so an
# injury/G-League gap isn't mistaken for a valid lag.
next_season_rows["PROJ_PREV_TS_PCT_1"] = next_season_rows["TS_PCT"]
has_lag_2 = (next_season_rows["SEASON_START_YEAR"] - next_season_rows["SEASON_LAG_1"]) == 1
next_season_rows["PROJ_PREV_TS_PCT_2"] = next_season_rows["TS_PCT_LAG_1"].where(has_lag_2)
has_lag_3 = (next_season_rows["SEASON_START_YEAR"] - next_season_rows["SEASON_LAG_2"]) == 2
next_season_rows["PROJ_PREV_TS_PCT_3"] = next_season_rows["TS_PCT_LAG_2"].where(has_lag_3)

next_season_rows["PROJECTED_AGE"] = next_season_rows["AGE"] + 1
next_season_rows["PROJ_USG_PCT"] = next_season_rows["USG_PCT"]

print(
    f"{len(next_season_rows)} of {proj_source['PLAYER_ID'].nunique()} active players "
    f"get a next-season projection (every player active in {int(latest_season_year)}, "
    f"regardless of career length)"
)

In [ ]:
projection_results = []
for _, row in next_season_rows.iterrows():
    prev2 = row["PROJ_PREV_TS_PCT_2"] if pd.notna(row["PROJ_PREV_TS_PCT_2"]) else None
    prev3 = row["PROJ_PREV_TS_PCT_3"] if pd.notna(row["PROJ_PREV_TS_PCT_3"]) else None
    samples = project_ts_pct_hier(
        age_value=row["PROJECTED_AGE"],
        usg_pct_value=row["PROJ_USG_PCT"],
        prev_ts_pct_1=row["PROJ_PREV_TS_PCT_1"],
        prev_ts_pct_2=prev2,
        prev_ts_pct_3=prev3,
        player_id=row["PLAYER_ID"],
    )
    hdi = az.hdi(samples, hdi_prob=0.94)
    projection_results.append({
        "PLAYER_ID": row["PLAYER_ID"],
        "PLAYER_NAME": row["PLAYER_NAME"],
        "PROJECTED_AGE": row["PROJECTED_AGE"],
        "PROJECTED_TS_PCT": samples.mean(),
        "HDI_LOW": hdi[0],
        "HDI_HIGH": hdi[1],
    })

projections_df = (
    pd.DataFrame(projection_results)
    .sort_values("PROJECTED_TS_PCT", ascending=False)
    .reset_index(drop=True)
)

os.makedirs("bayesian_ar_output", exist_ok=True)
projections_df.to_csv("bayesian_ar_output/next_season_projections.csv", index=False)

projections_df.head(15)

In [ ]:
from dashboard import build_dashboard

dashboard_path = build_dashboard()
print(f"Dashboard written to {dashboard_path} ({len(projections_df)} players)")

### Reading the dashboard

Open `bayesian_ar_output/dashboard.html` in a browser (it's a self-contained
static file -- no server needed), or run `python dashboard.py` from the
repo root, which rebuilds it from
`bayesian_ar_output/next_season_projections.csv` and opens it for you. It's
sorted by projected TS% descending by default; click a column header to
re-sort, or use the search box to filter to a specific player. Each bar
shows the posterior mean on a shared scale across all players, with a
shaded band for the 94% credible interval -- a wide band means the model
is uncertain about that player specifically (usually a smaller or noisier
recent sample, including anyone with fewer than three lags available), not
that the point estimate is unreliable in some other way.

This ranks by the *posterior mean* alone. Two players can have the same
mean with very different HDI widths -- if the decision this feeds into
cares about downside risk (e.g., a roster spot, not just an offseason
talking point), sort or filter by `HDI_LOW` instead, not the mean.